# Agentic RAG — Phase 3

Phase 1 built a RAG *chain*: one query → one retrieval → one answer. A chain can't recover from a
bad first retrieval. This notebook turns it into an **agent** that runs a loop:

```
plan → search → read → "is this enough?" → refine & search again → answer with citations (or abstain)
```

The retrieve → **self-critique** → retry loop is the thing a chain fundamentally can't do.

### The setup
- **Brain:** **Claude** (`claude-opus-4-8`) via the **tool-use API**, driven by a **ReAct loop you
  write by hand** — so you understand the mechanics, not just a framework call.
- **Retrieval:** local (mpnet embeddings + Qdrant + reranker), exposed to Claude as a
  `search_filings` **tool**.
- **No local LLM this time** — Claude is the reasoner, so there's **no Mistral, no GPU, no OOM**.
  A plain **CPU** Colab runtime is fine.

### The new evaluation surface (your edge)
An agent is harder to evaluate than a chain: you grade the *behavior*, not just the answer —
did it call the right tool, re-query when the context was thin, cite real sources, and abstain when
the filings don't contain the answer? That's trajectory + tool-use eval, and almost nobody instruments it.

> **Runtime:** Colab, **CPU is fine** (Runtime → Change runtime type → CPU). Secret (key icon):
> `ANTHROPIC_API_KEY`. `HF_TOKEN` is not needed here. Run top-to-bottom once.

## Setup

In [1]:
!pip install -qU langchain langchain-community langchain-huggingface langchain-qdrant
!pip install -qU qdrant-client sentence-transformers beautifulsoup4 lxml anthropic

In [2]:
import os, re, time, textwrap
import requests
import numpy as np
from bs4 import BeautifulSoup
from google.colab import userdata

os.environ["ANTHROPIC_API_KEY"] = userdata.get('ANTHROPIC_API_KEY')

import anthropic
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore
from sentence_transformers import CrossEncoder

SEC_USER_AGENT = "David Schaaf davidschaaf@berkeley.edu"   # edit to your name/email
AGENT_MODEL = "claude-opus-4-8"
client = anthropic.Anthropic()
print("Setup ready. Agent brain:", AGENT_MODEL)

Setup ready. Agent brain: claude-opus-4-8


## Rebuild retrieval *(provided — Phases 1–2)*

Fetch → clean → chunk (each chunk keeps a stable `chunk_id` we'll cite) → embed → index → reranked
retrieval. This is the *tool* the agent will call. No LLM loads here.

In [3]:
COMPANIES = {"AAPL": 320193, "MSFT": 789019, "NVDA": 1045810}

def fetch_latest_10k_html(ticker, cik):
    headers = {"User-Agent": SEC_USER_AGENT}
    recent = requests.get(f"https://data.sec.gov/submissions/CIK{cik:010d}.json",
                          headers=headers).json()["filings"]["recent"]
    i = next(j for j, f in enumerate(recent["form"]) if f == "10-K")
    acc, doc = recent["accessionNumber"][i].replace("-", ""), recent["primaryDocument"][i]
    html = requests.get(f"https://www.sec.gov/Archives/edgar/data/{cik}/{acc}/{doc}",
                        headers=headers).text
    time.sleep(0.5)
    return {"ticker": ticker, "html": html}

def clean_filing_text(html):
    soup = BeautifulSoup(html, "lxml")
    for t in soup(["script", "style"]):
        t.decompose()
    return re.sub(r"\s+", " ", soup.get_text(" ")).strip()

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
documents = []
for tk, cik in COMPANIES.items():
    print("Fetching", tk, "...")
    f = fetch_latest_10k_html(tk, cik)
    for chunk in splitter.split_text(clean_filing_text(f["html"])):
        documents.append(Document(page_content=chunk,
            metadata={"chunk_id": len(documents), "ticker": tk}))

embeddings = HuggingFaceEmbeddings(model_name="all-mpnet-base-v2")
qdrant = QdrantVectorStore.from_documents(
    documents, embeddings, location=":memory:", collection_name="sec_agent")
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def retrieve_reranked(query, fetch_k=10, top_k=4):
    cands = qdrant.similarity_search(query, k=fetch_k)
    scores = reranker.predict([(query, d.page_content) for d in cands])
    return [d for _, d in sorted(zip(scores, cands), key=lambda x: x[0], reverse=True)][:top_k]

print(f"Retrieval ready over {len(documents)} chunks.")

Fetching AAPL ...


/tmp/ipykernel_8155/1885630247.py:15: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(html, "lxml")


Fetching MSFT ...
Fetching NVDA ...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Retrieval ready over 1093 chunks.


---
## 1 · Give the agent a tool

An agent acts by calling **tools** you define. Here there's one: `search_filings`. You describe it
to the model with a JSON-schema definition (name, description, inputs); the model decides *when* and
*with what query* to call it. The `description` is prompt-engineering — it's how the model learns
what the tool is for.

In [4]:
# --- PROVIDED: the tool's execution (wraps your retriever; tags each hit with its id) ---
def do_search(query, k=4):
    docs = retrieve_reranked(query, top_k=k)
    ids = [d.metadata["chunk_id"] for d in docs]
    text = "\n\n".join(
        f"[id={d.metadata['chunk_id']} | {d.metadata['ticker']}] {d.page_content}" for d in docs)
    return text, ids

_t, _ids = do_search("What are Apple's main risk factors?")
print("returned ids:", _ids, "\n", _t[:200], "...")

returned ids: [45, 48, 62, 123] 
 [id=45 | AAPL] by reference into this filing. Further, the Company’s references to website URLs are intended to be inactive textual references only. Apple Inc. | 2025 Form 10-K | 4 Item 1A. Risk Facto ...


In [11]:
SEARCH_TOOL = {
    "name": 'search_filings',
    "description": "Search SEC 10-K filings for Apple, Microsoft, and NVIDIA. Returns a list of related document chunks for the query.",
    "input_schema": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string"
            }
        },
        "required": ["query"]
    }
}

**🎤 Talk-track — Tools.** *"A tool is just a typed contract between the model and my code —
a schema the model plans against and an executor I control. The description is where the behavior
lives: 'call again with a refined query if results are thin' is what turns a one-shot lookup into an
agent that self-corrects."*

---
## 2 · The ReAct loop *(the centerpiece)*

This is the agent. The pattern: call the model with the tools available; if it returns a
`tool_use` block, **execute the tool, feed the result back, and loop**; stop when it answers
(`stop_reason == "end_turn"`). A step cap prevents runaway loops. The self-correction is emergent —
the system prompt tells Claude to judge sufficiency and re-query, and the loop lets it.

In [33]:
SYSTEM = (" ".join([
    "You answer questions about SEC 10-K filings for Apple, Microsoft, and NVIDIA.",
    "Use the search_filings tool to gather evidence.",
    "If the returned passages do not fully answer the question, refine your query and search again (a few times if needed). ",
    "Answer ONLY from retrieved passages, and cite every claim with its source id in brackets e.g. [42].",
    "If the filings do not contain the answer, say you don't have that information rather than guessing.",
    "Be as concise as possible (2 paragraphs max)."
]))

MAX_STEPS = 5
print("System prompt + step cap set.")

System prompt + step cap set.


In [26]:
# --- YOU WRITE: the hand-rolled ReAct loop ---
# Loop up to MAX_STEPS:
#   1) client.messages.create(model=AGENT_MODEL, max_tokens=1024, system=SYSTEM,
#          tools=[SEARCH_TOOL], messages=messages)
#   2) if resp.stop_reason != "tool_use": break  (the model has answered)
#   3) append the assistant turn: {"role":"assistant","content": resp.content}
#   4) for each block b in resp.content where b.type == "tool_use":
#          run do_search(**b.input); log b.input["query"] and the returned ids;
#          collect {"type":"tool_result","tool_use_id": b.id, "content": <text>}
#   5) append {"role":"user","content": <all tool_results>}
# Return {"answer", "searches", "retrieved_ids"} (answer = joined text blocks of the last resp).
def run_agent(question):
    ### YOUR CODE HERE
    question_prompt = {"role": "user", "content": question}
    messages = [question_prompt]
    queries = []
    document_ids = []

    for _ in range(MAX_STEPS):
      resp = client.messages.create(model=AGENT_MODEL, max_tokens=1024, system=SYSTEM,
                           tools=[SEARCH_TOOL], messages=messages)
      print(resp)
      if resp.stop_reason != "tool_use":
          break
      messages.append({"role": "assistant", "content": resp.content})

      tool_results = []
      for block in resp.content:
        if block.type == "tool_use":
          query = block.input["query"]
          queries.append(query)
          text, ids = do_search(query)
          document_ids.extend(ids)
          tool_results.append({"type": "tool_result", "tool_use_id": block.id, "content": text})

      print(tool_results)
      messages.append({"role": "user", "content": tool_results})

    text_blocks = filter(lambda x: x.type == "text", resp.content)
    response_text = [x.text for x in text_blocks]
    return {"answer": "\n".join(response_text), "searches": queries, "retrieved_ids": list(set(document_ids))}

In [34]:
# --- INSTRUMENT: watch it plan, search, and (maybe) re-query ---
out = run_agent("How does NVIDIA describe competition, and what risks does it tie to supply?")
print("SEARCH TRAJECTORY:")
for i, q in enumerate(out["searches"], 1):
    print(f"  {i}. {q}")
print(f"\n({len(out['searches'])} search call(s); {len(set(out['retrieved_ids']))} unique chunks seen)\n")
print("ANSWER:\n", textwrap.fill(out["answer"], 100))

Message(id='msg_012okVRwc8VUyPZWYzRVHEtc', container=None, content=[TextBlock(citations=None, text="I'll search for information on how NVIDIA describes competition and supply-related risks in its 10-K filing.", type='text'), ToolUseBlock(id='toolu_01HjSV1XcPvFaf65XEH7D9YJ', caller=DirectCaller(type='direct'), input={'query': 'NVIDIA competition competitive market competitors'}, name='search_filings', type='tool_use'), ToolUseBlock(id='toolu_01USfaexy1kiuv4fzqXETbqn', caller=DirectCaller(type='direct'), input={'query': 'NVIDIA supply chain manufacturing risks dependence suppliers'}, name='search_filings', type='tool_use')], model='claude-opus-4-8', role='assistant', stop_details=None, stop_reason='tool_use', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='global', input_tokens=591, output_tokens=165, output_tokens_details=Output

**🎤 Talk-track — The loop.** *"This is ~15 lines and it's the whole agent: call, check the
stop reason, execute tools, feed results back, repeat. No framework magic — I can point at exactly
where a decision happens. The number of search calls is emergent: on an easy question it answers in
one hop; on a compound one it decomposes and searches twice."*

✍️ **Reflect:** what happens if you remove the "refine your query and search again" line from the
system prompt? (Predict, then try it — that one sentence is what makes it an agent.)

---
## 3 · Guardrail — are the citations real?

The agent is told to cite `[id]`. A guardrail *verifies* that: every cited id must actually appear
in what the agent retrieved. A citation to an id it never saw is a fabricated source — exactly the
kind of silent failure you want to catch automatically in a regulated setting.

In [ ]:
# --- YOU WRITE: extract cited ids and score their validity ---
# extract_cited_ids: regex the answer for [<digits>] -> a set of ints
# grounding_score: fraction of cited ids that are in retrieved_ids (None if nothing cited)
def extract_cited_ids(answer):
    ### YOUR CODE HERE
    pass

def grounding_score(cited_ids, retrieved_ids):
    ### YOUR CODE HERE
    pass

cited = extract_cited_ids(out["answer"])
print("cited ids:", cited)
print("grounding score:", grounding_score(cited, out["retrieved_ids"]))

**🎤 Talk-track — Guardrail.** *"'Cite your sources' in a prompt is a suggestion; verifying
that the cited ids were actually retrieved is a control. It's a cheap, deterministic check that
turns a hallucinated citation into a caught error instead of a shipped one."*

---
## 4 · Evaluate the *behavior*, not just the answer

Phase 2 graded answers. An agent needs **trajectory / tool-use** eval too. Over a small question
set — including one whose answer is **not in the filings** — we measure: how many searches it runs,
whether it grounds its citations, and whether it **abstains** when it should.

In [ ]:
# --- PROVIDED: a small eval set (last one is deliberately unanswerable from the filings) ---
QUESTIONS = [
    "What are the main risk factors NVIDIA highlights?",
    "How does Microsoft describe competition in its industry?",
    "What does Apple say about its revenue?",
    "What was NVIDIA's stock price on June 1st, 2026?",   # not in a 10-K -> should abstain
]
ABSTAIN_MARKERS = ("don't have", "do not have", "not contain", "cannot find", "no information")
print(len(QUESTIONS), "eval questions.")

In [ ]:
# --- YOU WRITE: run the agent over the set and summarize its behavior ---
# For each question: run_agent, then record n_searches, grounding_score of its citations,
# and whether it abstained (any ABSTAIN_MARKERS substring in the lowercased answer).
def evaluate_agent(questions):
    ### YOUR CODE HERE
    pass

rows = evaluate_agent(QUESTIONS)

In [ ]:
# --- INSTRUMENT: the agentic scorecard ---
answerable = rows[:-1]
print(f"{'question':<52}{'searches':>9}{'grounding':>11}{'abstain':>9}")
for r in rows:
    g = "-" if r["grounding"] is None else f"{r['grounding']:.2f}"
    print(f"{r['question'][:50]:<52}{r['n_searches']:>9}{g:>11}{str(r['abstained']):>9}")

grounded = [r['grounding'] for r in answerable if r['grounding'] is not None]
print(f"\nMean searches/question:      {np.mean([r['n_searches'] for r in rows]):.1f}")
print(f"Answerable citing sources:   {len(grounded)}/{len(answerable)}"
      f"  (mean grounding {np.mean(grounded):.2f})" if grounded else "")
print(f"Abstained on the unanswerable question: {rows[-1]['abstained']}")

**🎤 Talk-track — Agentic eval.** *"For an agent I measure the trajectory, not just the
final string: does it search the right number of times, ground its citations, and refuse when the
evidence isn't there? Abstention is the one everybody forgets — a system that never says 'I don't
know' is a liability, especially in finance. This is the eval most teams don't have, and it's where
I add the most value."*

---
## Recap — from chain to agent

| Piece | What it adds over the Phase-1 chain |
|-------|--------------------------------------|
| **Tool** | The model *decides* when and what to retrieve, instead of a fixed single call |
| **ReAct loop** | Retrieve → self-critique → **re-query** → answer; recovers from a bad first hit |
| **Grounding guardrail** | Fabricated citations become caught errors, not shipped ones |
| **Agentic eval** | Grades *behavior* — searches, grounding, abstention — not just the answer |

You've now built the full arc: **working pipeline (P1) → measured pipeline (P2) → agent that acts,
self-corrects, and is evaluated on its behavior (P3).**

**What production would add next:** parallel tool calls and more tools (a figures/XBRL tool, a
cross-company comparator); serving the agent behind an API with per-step logging; online trajectory
monitoring and drift alerts; and reusing the Phase-2 Claude judge to score final-answer faithfulness
alongside these behavioral metrics.